# Variables de clasificación · MobileNetV2 (issue #35)

Estudio de una variable por corrida: augmentation, pesos de la loss, LR y early stopping.
La referencia es `mobilenetv2/imagenet_finetuning` con `comun_v2` ya guardada en Drive;
**no se vuelve a entrenar ni se modifica el notebook 04**. Las siete variantes usan
`variables_v1` y quedan fuera de la comparación de arquitecturas.

Abrir una sesión nueva de Colab con GPU T4 y ejecutar todo. Para analizar corridas
existentes sin GPU, poner `ENTRENAR = False`: no se carga TensorFlow ni se escribe
ningún resultado. Este notebook tiene permiso de entrenamiento al montar Drive;
para una revisión estrictamente de solo lectura usar el notebook 10.

Solo se saltean modos completos cuya configuración coincide. Un modo incompleto
comienza de nuevo; no reanuda épocas. `REPETIR = True` archiva, no borra.

**Val (162 imágenes) elige checkpoints y también reporta; las métricas son optimistas.**
Jamaica tiene solo 23 imágenes. Un IC95% de diferencia que incluya cero no permite
concluir mejora. Los intervalos no corrigen selección de checkpoints, correlación
espacial ni las múltiples comparaciones de este estudio. No modificar `comun_v2`
a partir de este análisis sin acordarlo en #15. N06/N16 mantienen referencia provisional.

In [ ]:
ENTRENAR = True
REPETIR = False

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    dependencias = "requirements-training.txt" if ENTRENAR else "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / dependencias)], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import aumentacion, clasificacion, entorno, datos, resultados, split
rutas = entorno.preparar()

from pdi_tp import comparacion, estudios
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Receta y siete variantes

`BASE` reproduce el modo fine-tuning del 04. Cada entrada pisa un único hiperparámetro.
En `sin_early_stopping` se completan 30 + 20 épocas; se sigue seleccionando el mejor
checkpoint de cada etapa por val_loss, igual que en la referencia. Para restaurar
los mejores pesos en memoria se usa el callback de Keras con paciencia mayor que
el total de épocas: no puede detener esta variante antes de completar 30 + 20.
Los checkpoints se escriben primero en el disco local de Colab y se copian a Drive
al guardar la corrida. No se leen checkpoints desde Drive entre etapas. Las variantes
de pesos usan `sample_weight`, nunca además `class_weight`. Val tiene peso 1.

In [ ]:
MODELO = "mobilenetv2"
BASE = {'modelo': 'MobileNetV2',
 'receta': 'comun_v2',
 'pesos': 'imagenet',
 'modo': 'fine_tuning',
 'img_size': 256,
 'batch_size': 32,
 'augmentation': True,
 'dropout': 0.3,
 'class_weight': False,
 'sample_weight': 'igual_masa_por_fuente_x_clase',
 'epocas_cabeza': 30,
 'lr_cabeza': 0.001,
 'epocas_fine_tuning': 20,
 'lr_fine_tuning': 1e-05,
 'epocas_completo': 60,
 'lr_completo': 0.001,
 'paciencia': 10,
 'inicio_early_stopping': 0,
 'momentum_batchnorm': None,
 'reducir_lr': False,
 'reducir_lr_factor': 0.5,
 'reducir_lr_paciencia': 3,
 'umbral': 0.5,
 'semilla': 42}

EXPERIMENTOS = {
    "sin_aug": {"augmentation": False},
    "sin_pesos": {"sample_weight": None},
    "pesos_clase": {"sample_weight": "clase"},
    "lr_ft_1e-4": {"lr_fine_tuning": 1e-4},
    "lr_ft_1e-6": {"lr_fine_tuning": 1e-6},
    "lr_cabeza_1e-4": {"lr_cabeza": 1e-4},
    "sin_early_stopping": {"early_stopping": False},
}
BASE["early_stopping"] = True
REFERENCIA = "mobilenetv2/imagenet_finetuning"
CONFIGURACIONES = {}
for nombre, cambios in EXPERIMENTOS.items():
    if len(cambios) != 1:
        raise ValueError("Cada variante debe cambiar exactamente una variable.")
    variable = next(iter(cambios))
    config = {**BASE, **cambios}
    estudios.validar_control(BASE, config, [variable])
    CONFIGURACIONES[nombre] = {**config, "receta": "variables_v1", "variable": variable}

# No se puede atribuir el cambio a una variable si la referencia tiene otra receta.
carpeta_ref = rutas.resultados_equipo / REFERENCIA
referencia_disponible = resultados.corrida_completa(carpeta_ref)
if referencia_disponible:
    params_ref = json.loads((carpeta_ref / "config.json").read_text())["params"]
    efectiva = {c: params_ref[c] for c in BASE if c in params_ref}
    efectiva.setdefault("early_stopping", True)  # el 04 siempre lo aplica
    estudios.validar_control(BASE, efectiva, [])
elif ENTRENAR:
    raise RuntimeError("Falta la referencia comun_v2 completa en Drive; correr primero el notebook 04.")
else:
    print("Falta la referencia: se mostrarán las variantes disponibles, sin diferencias pareadas.")

a_correr = {}
for nombre, config in CONFIGURACIONES.items():
    carpeta = resultados.ruta_corrida(rutas, MODELO, nombre)
    if resultados.corrida_completa(carpeta) and not REPETIR:
        guardada = json.loads((carpeta / "config.json").read_text())["params"]
        estudios.validar_control(config, guardada, [])
        print("Completo, se saltea:", nombre)
    elif ENTRENAR:
        a_correr[nombre] = config
print("A entrenar:", list(a_correr))

## Datos y funciones de entrenamiento (misma arquitectura y etapas que el 04)

In [ ]:
if ENTRENAR:
    import time
    import tensorflow as tf
    import keras
    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("Activar GPU T4 para entrenar, o ENTRENAR=False para leer resultados.")
    archivos = split.cargar_split(rutas)
    X_train = np.stack([datos.leer_imagen(p) for p in archivos["train"]])
    X_val = np.stack([datos.leer_imagen(p) for p in archivos["val"]])
    y_train = np.array([datos.tiene_paneles(datos.label_de(rutas, p)) for p in archivos["train"]], dtype="float32")
    y_val = np.array([datos.tiene_paneles(datos.label_de(rutas, p)) for p in archivos["val"]], dtype="float32")
    fuentes_train = np.array([split.fuente(p.stem) for p in archivos["train"]])
    PESOS_VAL = np.ones(len(y_val), dtype="float32")
    AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def aumentar(x):
    return aumentacion.google_maps(x)


def armar_ds(X, y, pesos, entrenamiento, config):
    ds = tf.data.Dataset.from_tensor_slices((X, y, pesos)).map(
        lambda x, y, peso: (tf.cast(x, tf.float32), y, peso))
    if entrenamiento:
        ds = ds.shuffle(len(X), seed=config["semilla"])
        if config["augmentation"]:
            ds = ds.map(lambda x, y, peso: (aumentar(x), y, peso), num_parallel_calls=AUTOTUNE)
    return ds.batch(config["batch_size"]).prefetch(AUTOTUNE)



def armar_modelo(config):
    s = config["img_size"]
    base = keras.applications.MobileNetV2(input_shape=(s, s, 3), include_top=False, weights=config["pesos"])
    if config["momentum_batchnorm"] is not None:
        for capa in base.layers:
            if isinstance(capa, keras.layers.BatchNormalization):
                capa.momentum = config["momentum_batchnorm"]

    entradas = keras.Input(shape=(256, 256, 3))
    x = keras.layers.Resizing(s, s)(entradas) if s != 256 else entradas
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x, training=False) if config["pesos"] == "imagenet" else base(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(config["dropout"])(x)
    salidas = keras.layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(entradas, salidas, name=config["modelo"]), base

In [ ]:
# Mantener pendientes al redefinir funciones: reintentar el guardado no vuelve a entrenar.
PENDIENTES_GUARDADO = globals().get("PENDIENTES_GUARDADO", {})


def guardar_pendiente(nombre):
    pendiente = PENDIENTES_GUARDADO[nombre]
    # No crear una carpeta local que imite Drive si el montaje perdió acceso.
    referencia = rutas.resultados_equipo / REFERENCIA / "config.json"
    if not referencia.is_file():
        raise FileNotFoundError(
            "No se ve la referencia en Drive. Conservar esta sesión, recuperar el acceso "
            "con entorno.preparar() y reintentar guardar_pendiente(nombre), sin entrenar.")
    carpeta = resultados.carpeta_corrida(rutas, MODELO, nombre, reiniciar=False)
    pendiente["modelo"].save(carpeta / "modelo.keras")
    pendiente["historia"].to_csv(carpeta / "historia.csv", index=False)
    # Copiar los checkpoints locales recién al publicar la corrida completa en Drive.
    import shutil
    for checkpoint in pendiente.get("checkpoints", []):
        shutil.copy2(checkpoint, carpeta / checkpoint.name)
    resultados.guardar_corrida(rutas, carpeta, archivos=pendiente["archivos"],
        probabilidad=pendiente["probabilidad"], params=pendiente["config"],
        umbral=pendiente["config"]["umbral"], metricas_extra=pendiente["extra"])
    del PENDIENTES_GUARDADO[nombre]  # liberar el modelo solo después del guardado completo


def entrenar_etapa(modelo, etapa, lr, epocas, train_ds, val_ds, config, carpeta):
    modelo.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")])
    checkpoint = carpeta / f"checkpoint_{etapa}.keras"
    callbacks = [keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss", save_best_only=True)]
    # Una paciencia mayor al total permite completar todas las épocas y, al terminar,
    # restaurar la mejor val_loss desde memoria, sin volver a leer un modelo de Drive.
    paciencia_efectiva = config["paciencia"] if config["early_stopping"] else epocas + 1
    callbacks.append(keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia_efectiva,
        restore_best_weights=True, start_from_epoch=config["inicio_early_stopping"]))
    historial = modelo.fit(train_ds, validation_data=val_ds, epochs=epocas, callbacks=callbacks, verbose=2)
    return pd.DataFrame(historial.history).assign(etapa=etapa, epoca=lambda t: np.arange(1, len(t) + 1))


def correr_experimento(nombre, config):
    if nombre in PENDIENTES_GUARDADO:
        estudios.validar_control(config, PENDIENTES_GUARDADO[nombre]["config"], [])
        guardar_pendiente(nombre)
        return
    carpeta = resultados.carpeta_corrida(rutas, MODELO, nombre, reiniciar=REPETIR)
    keras.backend.clear_session()
    keras.utils.set_random_seed(config["semilla"])
    pesos = clasificacion.elegir_pesos_muestra(fuentes_train, y_train, config["sample_weight"])
    train_ds = armar_ds(X_train, y_train, pesos, entrenamiento=True, config=config)
    val_ds = armar_ds(X_val, y_val, PESOS_VAL, entrenamiento=False, config=config)
    modelo, base = armar_modelo(config)
    # El disco de Colab evita depender del montaje de Drive entre épocas/etapas.
    carpeta_local = repo / "resultados_locales" / "checkpoints_variables" / nombre
    carpeta_local.mkdir(parents=True, exist_ok=True)
    inicio = time.perf_counter()
    base.trainable = False
    cabeza = entrenar_etapa(modelo, "cabeza", config["lr_cabeza"], config["epocas_cabeza"],
                           train_ds, val_ds, config, carpeta_local)
    base.trainable = True
    for capa in base.layers:
        if isinstance(capa, keras.layers.BatchNormalization):
            capa.trainable = False
    ajuste = entrenar_etapa(modelo, "fine_tuning", config["lr_fine_tuning"], config["epocas_fine_tuning"],
                           train_ds, val_ds, config, carpeta_local)
    segundos = time.perf_counter() - inicio
    probabilidad = modelo.predict(val_ds, verbose=0).ravel()
    PENDIENTES_GUARDADO[nombre] = dict(modelo=modelo, config=dict(config),
        probabilidad=probabilidad.copy(), archivos=list(archivos["val"]),
        historia=pd.concat([cabeza, ajuste], ignore_index=True),
        checkpoints=[carpeta_local / f"checkpoint_{etapa}.keras" for etapa in ("cabeza", "fine_tuning")],
        extra={"segundos_entrenamiento": segundos,
               "epocas_por_etapa": {"cabeza": len(cabeza), "fine_tuning": len(ajuste)}})
    guardar_pendiente(nombre)


In [ ]:
if ENTRENAR:
    for nombre, config in a_correr.items():
        carpeta = resultados.ruta_corrida(rutas, MODELO, nombre)
        if nombre not in PENDIENTES_GUARDADO and resultados.corrida_completa(carpeta) and not REPETIR:
            guardada = json.loads((carpeta / "config.json").read_text())["params"]
            estudios.validar_control(config, guardada, [])
            print("Completo, se saltea:", nombre)
            continue
        print("Experimento:", nombre, "· variable:", config["variable"])
        correr_experimento(nombre, config)


### Si falla el guardado al terminar las épocas

No reiniciar el entorno ni cambiar `REPETIR` a True. `PENDIENTES_GUARDADO` conserva
el modelo, las probabilidades, la configuración y el historial **en memoria de esta
sesión** hasta guardar todo. Los checkpoints temporales quedan en
`/content/PDI-TP/resultados_locales/checkpoints_variables/<modo>/` y se copian a la
carpeta común de Drive al finalizar. Ninguno de estos rescates sobrevive a perder
la máquina de Colab: guardar la corrida en Drive antes de terminar la sesión.

1. Comprobar el acceso a la carpeta compartida; si se perdió, ejecutar `rutas = entorno.preparar()`.
2. Reintentar solo el guardado con `guardar_pendiente("sin_aug")` (usar el modo que falló).
   Este paso vuelve a comprobar Drive y crea la carpeta del modo si falta, sin archivar.
3. Ejecutar la celda del bucle para continuar con los modos pendientes. Con `REPETIR=False`
   vuelve a comprobar cuáles ya están completos antes de entrenar.

No ejecutar todas las celdas ni reiniciar mientras haya un guardado pendiente.
Las funciones se pueden redefinir sin borrar `PENDIENTES_GUARDADO`.


## Métricas reales en val, Jamaica y por fuente

Las tablas leen exclusivamente el formato común; se informa qué falta. Las diferencias
son variante menos referencia, con el umbral fijo de 0.5. Los pesos por fuente se
inspeccionan también por fuente: una fuente de una sola clase no tiene AUC definido.
**Val seleccionó checkpoints y también reporta.**

In [ ]:
inventario = resultados.cargar_corridas(rutas)
corridas = [REFERENCIA] if referencia_disponible else []
for nombre, config in CONFIGURACIONES.items():
    carpeta = rutas.resultados_equipo / MODELO / nombre
    if resultados.corrida_completa(carpeta):
        guardada = json.loads((carpeta / "config.json").read_text())["params"]
        estudios.validar_control(config, guardada, [])
        corridas.append(f"{MODELO}/{nombre}")
    else:
        print("Pendiente de entrenar/completar:", nombre)
if corridas:
    display(estudios.tabla_metricas(rutas.resultados_equipo, corridas).round(4))
    pesos = [c for c in corridas if c == REFERENCIA or c.split("/")[1] in ("sin_pesos", "pesos_clase")]
    display(estudios.tabla_metricas(rutas.resultados_equipo, pesos, por_fuente=True).round(4))
if referencia_disponible and len(corridas) > 1:
    predicciones = resultados.cargar_predicciones(rutas)
    for jamaica in (False, True):
        print("Jamaica (23)" if jamaica else "Val (162)", "· variante − referencia · IC95%")
        tablas = [comparacion.diferencias(predicciones, corridas, REFERENCIA, metrica=m,
                  solo_jamaica=jamaica, n=2000, semilla=42) for m in estudios.METRICAS]
        display(pd.concat(tablas, ignore_index=True).round(4))

## Curvas de entrenamiento y validación

Se leen historiales guardados; no hace falta entrenar de nuevo. La línea vertical
separa cabeza y fine-tuning. Mirar si val_loss sube mientras train_loss baja, y si
el LR lleva a convergencia lenta o inestable. Estas curvas no demuestran una mejora
sin considerar las diferencias pareadas. Un historial ausente se informa.

In [ ]:
for corrida in corridas:
    ruta_historia = rutas.resultados_equipo / corrida / "historia.csv"
    if not ruta_historia.is_file():
        print("Sin historial:", corrida)
        continue
    historia = pd.read_csv(ruta_historia)
    figura, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
    desplazamiento = 0
    for etapa, grupo in historia.groupby("etapa", sort=False):
        epocas = np.arange(1, len(grupo) + 1) + desplazamiento
        for eje, metrica in zip(ejes, ("loss", "accuracy")):
            eje.plot(epocas, grupo[metrica], label=f"train {etapa}")
            eje.plot(epocas, grupo[f"val_{metrica}"], "--", label=f"val {etapa}")
            eje.set_title(metrica)
            eje.set_xlabel("época acumulada")
            if desplazamiento:
                eje.axvline(desplazamiento + 0.5, color="gray", lw=0.7)
        desplazamiento += len(grupo)
    ejes[0].legend(fontsize=8)
    figura.suptitle(corrida + " · val también selecciona checkpoint")
    figura.tight_layout()
    plt.show()
    plt.close(figura)

## Para completar el informe

Registrar fecha, commit, entorno y modos realmente ejecutados. No hay métricas
precargadas. Si un IC95% incluye cero, reportar que no se detectó una diferencia;
con una semilla tampoco se conoce la variación entre entrenamientos.
Mantener abiertos los pendientes experimentales del issue #35 hasta ejecutar las
siete corridas y revisar sus resultados.